# 📧 Detecting Spam Emails
## Notebook 4: Support Vector Machine (SVM)

**KKR Gen AI Innovations** | [kkrgenaiinnovations.com](https://kkrgenaiinnovations.com/)

---

### 🎯 What You Will Learn
1. How SVM works with an intuitive example
2. How to train an SVM model for text classification
3. How to compare SVM with Naive Bayes
4. How to tune the C parameter

---

### 📌 SVM Intuition

Imagine spam emails as red dots and ham emails as blue dots on a 2D graph.
SVM finds the best LINE (or hyperplane in higher dimensions) to separate them:

```
Red (Spam)  ●  ●  ●  ●
                        ← Maximum Margin
            ══════════ ← Decision Boundary
                        ← Maximum Margin  
Blue (Ham)  ○  ○  ○  ○
```

The key is **maximizing the margin** — the wider the gap, the more confident the model is.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.utils import download_dataset
from src.data_preprocessing import load_dataset, preprocess_dataframe
from src.feature_extraction import split_data, extract_tfidf_features
from src.models.svm_model import (
    train_svm, evaluate_svm, plot_confusion_matrix,
    predict_single_email
)

print('✅ All modules loaded!')

## Step 1: Load and Preprocess Data

In [ ]:
download_dataset('../data')
df = load_dataset('../data/spam.csv')
df = preprocess_dataframe(df)

X = df['cleaned_message']
y = df['label_encoded']
X_train, X_test, y_train, y_test = split_data(X, y)

X_train_tfidf, X_test_tfidf, vectorizer = extract_tfidf_features(
    X_train.tolist(), X_test.tolist()
)

print('\n✅ Data ready!')

## Step 2: Train the SVM Model

In [ ]:
# Train SVM with linear kernel
model = train_svm(X_train_tfidf, y_train, kernel='linear', C=1.0)
print('\n🎉 SVM trained!')

## Step 3: Evaluate the Model

In [ ]:
metrics = evaluate_svm(model, X_test_tfidf, y_test)

## Step 4: Confusion Matrix

In [ ]:
os.makedirs('../results', exist_ok=True)
plot_confusion_matrix(
    model, X_test_tfidf, y_test,
    save_path='../results/svm_confusion.png'
)

## Step 5: Understanding the C Parameter

**C** controls how strictly the model separates classes:
- **Small C (e.g., 0.1)**: More lenient — allows some misclassifications, smoother boundary
- **Large C (e.g., 10)**: More strict — tries to classify everything correctly

In [ ]:
from sklearn.metrics import accuracy_score

C_values = [0.01, 0.1, 1.0, 5.0, 10.0]
train_scores = []
test_scores = []

print('Testing different C values...')
for C in C_values:
    m = train_svm(X_train_tfidf, y_train, C=C)
    train_acc = accuracy_score(y_train, m.predict(X_train_tfidf))
    test_acc  = accuracy_score(y_test, m.predict(X_test_tfidf))
    train_scores.append(train_acc)
    test_scores.append(test_acc)
    print(f'  C={C:5.2f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}')

# Plot C parameter effect
plt.figure(figsize=(8, 5))
plt.plot(C_values, train_scores, 'b-o', label='Training Accuracy', linewidth=2)
plt.plot(C_values, test_scores, 'r-o', label='Test Accuracy', linewidth=2)
plt.xscale('log')
plt.xlabel('C Parameter (log scale)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of C Parameter on SVM Accuracy', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: Predict Custom Emails

In [ ]:
test_emails = [
    "CONGRATULATIONS! You've won a FREE iPhone! CLICK HERE to claim your prize NOW!",
    "Hey, are you free for lunch tomorrow? Let's catch up!",
    "URGENT: Your account needs verification. Send your password to confirm.",
    "Don't forget the team meeting at 3pm in Conference Room B.",
]

print('='*60)
print('📧 SVM SPAM DETECTION RESULTS')
print('='*60)

for email in test_emails:
    result = predict_single_email(model, vectorizer, email)
    status = '🚨 SPAM' if result['is_spam'] else '✅ HAM '
    print(f'{status} ({result["spam_confidence"]:.1%}) | "{email[:50]}..."')

## ✅ Summary

**SVM achieved ~98% accuracy** — better than Naive Bayes!

Key takeaways:
- More powerful than Naive Bayes for text
- The C parameter controls model complexity
- C=1.0 usually works well (try others!)
- Slower to train than Naive Bayes but still fast

---

### ➡️ Next Step
Open **Notebook 5: LSTM Neural Network** for the most powerful approach!

---
*KKR Gen AI Innovations | [kkrgenaiinnovations.com](https://kkrgenaiinnovations.com/)*